# SemIf / Qwen3.5-4B direct logit readout on Colab L4

This notebook is a thin reproducible wrapper around `scripts/run_experiment.py`. It measures model loading, first inference, warmup, steady-state throughput, and CUDA peak memory. It uses the pinned TheoLeeCJ/SemIf implementation and does not use the separate AlexWortega NLI head.

In [ ]:
from pathlib import Path
import subprocess

ROOT = Path.cwd()
if not (ROOT / 'scripts' / 'run_experiment.py').exists():
    candidate = Path('/content/jev-colab-lab/experiments/semif')
    if (candidate / 'scripts' / 'run_experiment.py').exists():
        ROOT = candidate
assert (ROOT / 'scripts' / 'run_experiment.py').exists(), f'Open this notebook from {ROOT}'
print(ROOT)

In [ ]:
subprocess.run(['uv', 'sync', '--extra', 'gpu', '--group', 'dev', '--frozen'], cwd=ROOT, check=True)
subprocess.run(['uv', 'run', '--extra', 'gpu', 'python', 'scripts/validate_fixture.py', 'data/questions.jsonl'], cwd=ROOT, check=True)

In [ ]:
output = ROOT / 'results' / 'semif-l4-notebook.json'
subprocess.run([
    'uv', 'run', '--extra', 'gpu', 'python', 'scripts/run_experiment.py',
    '--input', 'data/questions.jsonl', '--output', str(output),
    '--expected-gpu', 'L4', '--warmup', '2', '--repeats', '5',
], cwd=ROOT, check=True)
print(output)

In [ ]:
import json
result = json.loads(output.read_text(encoding='utf-8'))
print(json.dumps({
    'status': result['status'],
    'gpu': result.get('runtime', {}).get('gpu'),
    'load_seconds': result.get('metrics', {}).get('model_load_seconds'),
    'first_inference_seconds': result.get('metrics', {}).get('first_inference_wall_seconds'),
    'steady_state': result.get('metrics', {}).get('steady_state'),
    'peak_vram': result.get('metrics', {}).get('peak_vram'),
}, ensure_ascii=False, indent=2))